In [1]:
from vistiq.io import ImageWriterConfig, ImageWriter, ImageLoader, ImageLoaderConfig, unstack_image
from vistiq.utils import ArrayIteratorConfig, check_device, resolve_futures 
from vistiq.core import Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import FuncProcessor, FuncProcessorConfig, PreprocessFlow, PreprocessFlowConfig, ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, MicroSAMMerger, MicroSAMMergerConfig
from vistiq.segment import TiledSegmentationFlow, TiledSegmentationFlowConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector, AnalysisFlowConfig, AnalysisFlow, IoSMetricsCalculatorConfig
from vistiq.core import labels_to_masks
from vistiq.analysis.overlap import (
    OverlapCalculator,
    LabelOverlapCalculatorConfig,   # or BoxOverlapCalculatorConfig / MaskOverlapCalculatorConfig
    IoSMetricsCalculatorConfig,
    metrics_calculator_configs,
    region_map_from_dataframe,      # if using region maps from DataFrames
)
from vistiq.analysis import MatrixAggregator, MatrixAggregatorConfig, MatrixCombiner, MatrixCombinerConfig, SpatialScopeConfig
from vistiq.matrix.types import UPPER
from vistiq.matrix.types import UPPER_ND
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig
from vistiq.analysis import KnnAnalysis, KnnAnalysisConfig, RnnAnalysis, RnnAnalysisConfig
from vistiq.graph import GraphBuilder, GraphBuilderConfig, GraphQuery, GraphQueryConfig, GraphQueryFormatter, GraphQueryFormatterConfig
from vistiq.graph import HierarchyBuilderConfig

from prefect import flow, task
from prefect.task_runners import ProcessPoolTaskRunner
from prefect.futures import wait
from prefect.futures import resolve_futures_to_results
from scipy.ndimage import binary_dilation


import stackview
import os
import copy
import numpy as np
import math
import logging
import pandas as pd
import itertools

from typing import Any, List, Tuple
from pathlib import Path

2026-07-06 15:54:58,068 - INFO - No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


# Configure logger and check availability of accelerators

In [2]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

logger.info(f"Available Torch accelerators: {check_device()}")

2026-07-06 15:54:58,927 - INFO - Found mps device: Apple Metal (MPS)
2026-07-06 15:54:58,927 - INFO - Available Torch accelerators: mps


# Load image

In [3]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="../../cshl/two-lobes/Animal 1.lif"; scene_index=0
#path="../../cshl/Taylor_GWAS/high/189/dgrp.189.g1.animal1.lif"; scene_index=1
#path="../../cshl/Taylor_GWAS/high/189/dgrp.189.g1.animal2.lif"; scene_index=0
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

embedding_path = "../../embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [4]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=False,
    #substack="Z:20-50"
)
img, metadata = ImageLoader(ilc).run(path)
metadata

2026-07-06 15:54:59,332 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:54:59,386 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 15:54:59,399 - INFO - Loading image from: ../../cshl/two-lobes/Animal 1.lif
2026-07-06 15:55:01,236 - INFO - Scenes found: ('Series001', 'Series002', 'Series003')
2026-07-06 15:55:01,363 - INFO - Loaded image: ../../cshl/two-lobes/Animal 1.lif scene=0 -> shape=(3, 93, 512, 512) dtype=uint8, channel_names=['Scrib', 'EdU', 'Dpn']
2026-07-06 15:55:01,365 - INFO - Loaded image with shape: (3, 93, 512, 512), dtype: uint8
2026-07-06 15:55:01,367 - INFO - Finished in state Completed()


{'scene_index': 0,
 'dim_order': 'CZYX',
 'axes': ['C', 'Z', 'Y', 'X'],
 'channel_names': ['Scrib', 'EdU', 'Dpn'],
 'channel_axis': 0,
 'shape': (3, 93, 512, 512),
 'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
 'pixel_unit': 'um',
 'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
 'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)}

In [5]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

# Preprocess

In [6]:
tissue_ppcfg = PreprocessFlowConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
        
        #FuncProcessorConfig(
        #    func="numpy.sum", 
        #    kwargs={"axis":("Z")}, # Project all channels into one
        #    strict_axis=False,     # don't throw exception if the input is a single channel image already
        #    #dtype=np.uint16,
        #),


    ]
)
#tissue_img, tissue_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
#metadata, tissue_metadata, tissue_img.shape

In [7]:
#tissue_metadata = copy.deepcopy(tissue_metadata)
#tissue_metadata["channel_names"] = ["Lobe"]
# segment tissue
#tissue_labels = TiledSegmentationFlow(tsfcfg).run(tissue_img, metadata=tissue_metadata, workers=2, verbose=0)


In [8]:
#stackview.slice(tissue_labels)

# Configuration for 3D Tissue Segmentation

In [9]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)

rfcfg = RegionFilterConfig(
    filters=[
        #RangeFilterConfig(
        #    attribute="volume",
        #    range=(100000, np.inf),
        #),
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(2000, np.inf),
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-xz", 
            range=(2000, np.inf),
        ),
        RangeFilterConfig(
            attribute="cross_sectional_area-yz", 
            range=(2000, np.inf),
        ),
        RangeFilterConfig(
            attribute="aspect_ratio", 
            range=(0.5, 1.0),
        ),
    ]
)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
    tile_factor=(3,3),
    resize_factor=(0.25, 0.25),
    iou_threshold=0.5,
    consensus_threshold=0.75,
)

# Configuration for Region Analysis

In [10]:
racfg = RegionAnalyzerConfig(
    properties=["slice_annotations","volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe",
    map_axes=True,
)
ra = RegionAnalyzer(racfg)


# Configuration for Cell Segmentation

In [11]:
# specify preprocessing config for cells
cell_ppcfg = PreprocessFlowConfig(
    processors = [
        #DoGConfig(
        #    sigma_low=1, # 5, 
        #    sigma_high=2, #12, 
        #    normalize=True,
        #    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z focal plane and channel
        #)
    ]
)


In [12]:
# Specify segmentation config
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
    #gpu_fraction=0.3,
)

min_cell_radius = 2.0
max_cell_radius = 7.0
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="cross_sectional_area-xy", 
            range=(np.pi*min_cell_radius**2, np.pi*max_cell_radius**2)
        )
    ]
)

cell_sfcfg = SegmentationFlowConfig(
    segmenter = mscfg,
    region_filter = rfcfg,
)

In [13]:
@flow
def analyze_cells(labels: list[np.ndarray], metadata: list[dict[str, Any]]) -> list[pd.DataFrame]:
    print ([l.shape for l in labels])
    print ([m["channel_names"] for m in metadata])
    racfg = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe"
    )
    ra = RegionAnalyzer(racfg)

    measurements = ra.run.map(labels, metadata=metadata)

    cdcfg = CoincidenceDetectorConfig(
        method="ios",
        iterator_config=ArrayIteratorConfig(slice_def=()),
        mode="outline",
    )
    label_index_combinations = list(itertools.combinations(range(len(labels)), 2))
    l1 = [labels[c[0]] for c in label_index_combinations]
    l2 = [labels[c[1]] for c in label_index_combinations]
    sn = [(metadata[c[0]]["channel_names"][0], metadata[c[1]]["channel_names"][0]) for c in label_index_combinations]
    print (sn)
    #for la1, la2, sna in zip(l1,l2,sn): 
    cim = CoincidenceDetector(cdcfg).run.map(l1, l2, stack_names=sn)
    return measurements
 

In [14]:
acfg = AnalysisFlowConfig(
    region_analyzer = RegionAnalyzerConfig(
        properties=["volume", "centroid", "cross_sectional_area", "bbox", "aspect_ratio"],
        iterator_config = ArrayIteratorConfig(slice_def=()),
        output_type="dataframe",
        index_on="object_id",
        map_axes=True,
    ),
    #coincidence_detector = CoincidenceDetectorConfig(
    #    method=IoSMetricsCalculatorConfig(),
    #    iterator_config=ArrayIteratorConfig(slice_def=()),
    #    mode="outline",
    #),
    hierarchy_builder=HierarchyBuilderConfig(
        orphan_strategy="drop", #"group",
    ),
    overlap_calculator = LabelOverlapCalculatorConfig(
        metrics_calculators = [IoSMetricsCalculatorConfig()],
        output_type="dataframe",
        annotate=True,
        triangle=7,
    ),
    overlap_filter = ValueFilterConfig(
        ref_value=0.5,
        axis=0,
        operator=">",
        triangle=UPPER_ND, # has to be upper to align with downstream establishment of objeect hierarchies
        output="masked_values",
    ),
    overlap_aggregator = MatrixAggregatorConfig(
        operation="count",
        axis=1,
    ),
    knn_analysis = KnnAnalysisConfig(
        k=5,
        mode="homotypic",
        scope=SpatialScopeConfig(
            match={"channel": "Lobe"}
        )
    ),
    rnn_analysis = RnnAnalysisConfig(
        radius=25,
        mode="homotypic",
        scope=SpatialScopeConfig(
            match={"channel": "Lobe"}
        )
    ),
        
)

In [15]:
@flow
def full_pipeline(img_path, scene_index=0, outdir=".", embedding_path="embeddings"):
    # load image
    img, metadata = ImageLoader(ilc).run(img_path)
    
    # TISSUE - brain lobes
    # preprocess
    tissue_img, tissue_metadata = PreprocessFlow(tissue_ppcfg).run(img, metadata=metadata, workers=-1)
    # segment tissue
    tissue_labels = TiledSegmentationFlow(tsfcfg).run(tissue_img, metadata=tissue_metadata, workers=2, verbose=0)
    if tissue_labels.ndim == img.ndim-2:
        print ("USING 2D TISSUE MASK")
        print(f"tissue_labels.ndim={tissue_labels.ndim}, np.max(tissue_labels)={np.max(tissue_labels)}, tissue_labels.dtype={tissue_labels.dtype}, img.ndim={img.ndim}")
        fc = FuncProcessorConfig(
            func="numpy.repeat",
            args=[img.shape[1]],
            kwargs={"axis":(0,)}, # stck along Z axis
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            output_dims={"Z": img.shape[-3], "Y": img.shape[-2], "X": img.shape[-1]},
            dtype=np.uint16,
        )
        arr = tissue_labels[np.newaxis, :, :]
        print (arr.shape)
        tissue_labels, tissue_metadata = FuncProcessor(fc).run(arr, metadata=tissue_metadata)
        print (f"{tissue_labels.shape}, {tissue_labels.dtype}, np.max(tissue_labels)={np.max(tissue_labels)}, {tissue_metadata}")
    else:
        print ("USING 3D TISSUE MASK")
    tissue_metadata = copy.deepcopy(tissue_metadata)
    tissue_metadata["channel_names"] = ["Lobe"]


    # BRAIN - all tissue combined
    #tissue_masks = labels_to_masks(tissue_labels)
    brain_mask = (tissue_labels>0).astype("uint16")
    focal_plane = len(tissue_labels)//2 
    brain_mask[focal_plane] = binary_dilation(brain_mask[focal_plane], iterations=1)
    brain_label = (brain_mask * 1).astype("uint16")
    brain_metadata = copy.deepcopy(tissue_metadata)

    # binary mask from segmented lobes
    #brain_mask = (tissue_labels > 0).astype("uint16")
    dcfg = FuncProcessorConfig(
        func="scipy.ndimage.binary_dilation",
        kwargs={"iterations": 1},  # XY pixels to grow
        iterator_config=ArrayIteratorConfig(slice_def=(-2, -1)),  # each YX plane
        normalize=False,
    )
    #brain_dilated, brain_metadata = FuncProcessor(dcfg).run(
    #    brain_mask,
    #    metadata=copy.deepcopy(tissue_metadata),
    #)
    #brain_label = brain_mask# brain_dilated#np.where(brain_dilated, 65535, 0).astype("uint16")
    brain_metadata["channel_names"] = ["Brain"]
    
    # CELLS
    # preprocess
    preprocessed, preprocessed_metadata = PreprocessFlow(cell_ppcfg).run(img, metadata=metadata)
    # split channels
    channels, channel_metadata = unstack_image(preprocessed, preprocessed_metadata, axis=metadata["channel_axis"], strict=False)
    # segment each channel separately
    #cell_labels = SegmentationFlow(cell_sfcfg).mapped_run(channels, metadata=channel_metadata, workers=1)
    cell_labels = [SegmentationFlow(cell_sfcfg).run(ch, metadata=ch_meta) for ch, ch_meta in zip(channels, channel_metadata)] 
    
    # Analyze CELLS and TISSUE
    # analyze regions in each channel separately
    combined_labels = [brain_label, tissue_labels, *cell_labels]
    combined_metadata = [brain_metadata, tissue_metadata, *channel_metadata]
    measurements = AnalysisFlow(acfg).run(combined_labels, metadata=combined_metadata)
    # measurements = analyze_cells([*cell_labels, lobe_labels, brain_label], metadata=[*channel_metadata, c_metadata, b_metadata])

    # save labels
    if outdir is None:
        outdir = Path(img_path).parent
    fname_stem = Path(img_path).stem
    imc = ImageWriterConfig(overwrite=True)
    outpaths = [os.path.join(outdir, f'{fname_stem}.scene-{meta.get("scene_index","")}.tif') for meta in combined_metadata]
    # print (outpaths)
    ImageWriter(imc).run.map(combined_labels, outpaths, metadata=combined_metadata)
    
    # make sure to resolve the futures to results
    return (
        resolve_futures(combined_labels),
        resolve_futures(combined_metadata),
        resolve_futures(measurements),
    )


In [16]:
combined_labels, combined_metadata, measurements = full_pipeline(path, scene_index=scene_index, outdir=".", embedding_path=embedding_path)

2026-07-06 15:55:36,650 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flows/ "HTTP/1.1 200 OK"
2026-07-06 15:55:36,944 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/ "HTTP/1.1 201 Created"
2026-07-06 15:55:37,155 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a4c083-8ca5-792b-8000-49c2c6757a58/set_state "HTTP/1.1 201 Created"
2026-07-06 15:55:37,376 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a4c083-8ca5-792b-8000-49c2c6757a58 "HTTP/1.1 200 OK"
2026-07-06 15:55:37,462 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe0

Using apple MPS device.


2026-07-06 15:55:51,167 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-07-06 15:55:55,102 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='../../embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-07-06 15:55:55,102 - INFO - StackProcessor.run: received workers=2 (type: <class 'in

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-07-06 15:56:15,849 - INFO - Finished in state Completed()
2026-07-06 15:56:15,850 - INFO - Finished in state Completed()
2026-07-06 15:56:16,249 - INFO - Running Untiler with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3)
2026-07-06 15:56:16,249 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-07-06 15:56:16,326 - INFO - factor=(3, 3), array.shape=(18, 93, 399, 399), vs.shape=(3, 18, 93, 399, 133), hs.shape=(3, 3, 18, 93, 133, 133), untiled.shape=(9, 18, 93, 133, 133)
2026-07-06 15:56:16,329 - INFO - Finished in state Completed()
2026-07-06 15:56:16,579 - INFO - Running OverlapCalculator with config: classname='Configurable' package='v

...

2026-07-06 15:57:33,100 - INFO - Finished in state Completed()
2026-07-06 15:57:33,167 - INFO - Finished in state Completed()
2026-07-06 15:57:33,169 - INFO - Finished in state Completed()
2026-07-06 15:57:33,398 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a4c088-9587-755e-8000-554241b91b66/set_state "HTTP/1.1 201 Created"
2026-07-06 15:57:34,088 - INFO - Finished in state Completed()
2026-07-06 15:57:34,256 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/task_runs/ "HTTP/1.1 201 Created"
2026-07-06 15:57:34,335 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/filter "HTTP/1.1 200 OK"
2026-07-06 15:57:34,450 - INFO - HTTP Request: POST https://api.prefect

Using apple MPS device.


2026-07-06 15:57:37,391 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-07-06 15:57:41,181 - INFO - Running MicroSAMSegmenter with config: classname='Configurable' package='vistiq.core' version=None command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='processes' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' checkpoint=None embedding_path='../../embeddings' pred_iou_thresh=0.88 stability_score_thresh=0.95 box_nms_thresh=0.7 crop_nms_thresh=0.7 min_mask_region_area=0 output_mode='instance_segmentation' with_background=True device=None device_no=0 gpu_fraction=1.0
2026-07-06 15:57:41,181 - INFO - StackProcessor.run: received workers=-1 (type: <class 'i

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-07-06 15:58:01,884 - INFO - Finished in state Completed()
2026-07-06 15:58:01,892 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/logs/ "HTTP/1.1 202 Accepted"
2026-07-06 15:58:01,947 - INFO - Finished in state Completed()
2026-07-06 15:58:01,948 - INFO - Finished in state Completed()
2026-07-06 15:58:02,205 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a4c08a-e900-7a69-8000-d8694338c949/set_state "HTTP/1.1 201 Created"
2026-07-06 15:58:03,199 - INFO - Finished in state Completed()
2026-07-06 15:58:03,430 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/task_runs/ "HTTP/1.1 201 Created"
2026-07-06 15:58:03,642 - INFO - HTTP Request: POST https://api.prefect.clou

In [17]:
stackview.slice(np.concatenate(combined_labels, axis=-1))

In [34]:
#for k,v in measurements.items():
#    print (k, type(v))

# Query graph for object ancestor lineage

1. Subcellular cellular Dpn -> tissue lobe -> organ brain
2. Count descendants in each channel

In [36]:
dag = measurements["containment_graph"]

gqcfg = GraphQueryConfig(
    attributes=["descendant_counts", "ancestor_lineage"],
    filter_attribute="channel",
    filter_value="Lobe",
    include_attributes=["object_name", "label", "channel", "volume"],
    lineage_value_attribute="label",
)
gq = GraphQuery(gqcfg)
gqfmt = GraphQueryFormatter(GraphQueryFormatterConfig())
result = gq.run(dag, node=None)
df_counts = gqfmt.run(result, attribute="descendant_counts")
df_lineage = gqfmt.run(result, attribute="ancestor_lineage")
df = pd.concat([df_counts, df_lineage], axis=1)
df = df.loc[:, ~df.columns.duplicated()].sort_values(["label"])
df

2026-07-06 16:13:07,784 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 16:13:07,827 - INFO - HTTP Request: GET https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/admin/storage "HTTP/1.1 404 Not Found"
2026-07-06 16:13:07,835 - INFO - Summarizing graph with config: classname='Configurable' package='vistiq.core' version=None command_group=None label_attribute='object_name' group_attribute='channel' filter_attribute='channel' filter_value='Lobe' include_attributes=['object_name', 'label', 'channel', 'volume'] lineage_value_attribute='label' weight_attribute='ios' neighbor_analysis=None neighbor_k=None neighbor_radius=None source_filter=None target_filter=None source_nodes=None predecessor_match=None seed_nodes=None attributes=['descendant_counts', 'ancestor_lineage'

,object_name,channel,label,volume,count Scrib,count Dpn,count EdU,lineage Brain
object_id,,,,,,,,
1d6dcdda74614b24ac6fa57eb4321a3e,Lobe 1,Lobe,1,296992.595265,151,93,173,1
7a045e0e38db467a84e1b40bfe82f787,Lobe 2,Lobe,2,286420.504943,138,92,187,1


In [22]:
from pyvis.network import Network

net = Network(notebook=True, cdn_resources='in_line', bgcolor="#222222", font_color="white", select_menu=True)
net.barnes_hut()

# Convert the networkx object
net.from_nx(dag.raw)
neighbor_map = net.get_adj_list()

# add neighbor data to node hover data
for node in net.nodes:
    node["title"] = node["object_name"] #+ "\n" +"  Neighbors:\n" + "\n".join(neighbor_map[node["id"]])
    #node["value"] = len(neighbor_map[node["object_name"]])

# Render
net.show("nx_graph.html")

nx_graph.html


In [23]:
net.show_buttons(filter_=['physics'])

# Summary

In [32]:
# add binary +/- categorization for channel overlap (based on channel overlap count) 
features=measurements["region_analyzer_all"].sort_values(["channel","label"])
for ch in np.unique(features["channel"]):
    count_col = f"count {ch}"
    if count_col in features.columns.values:
        features[f"{ch} +"] = features[count_col]>0
features.columns.values

array(['bbox-start-z', 'bbox-start-y', 'bbox-start-x', 'bbox-end-z',
       'bbox-end-y', 'bbox-end-x', 'centroid-z', 'centroid-y',
       'centroid-x', 'label', 'aspect_ratio-yz', 'aspect_ratio-xz',
       'aspect_ratio-xy', 'aspect_ratio', 'cross_sectional_area-yz',
       'cross_sectional_area-xz', 'cross_sectional_area-xy', 'volume',
       'channel', 'stack_id', 'slice_id', 'object_name', 'count Dpn',
       'count EdU', 'count Lobe', 'count Scrib', 'lineage Brain',
       'lineage Dpn', 'lineage EdU', 'lineage Lobe', 'lineage Scrib',
       'knn_count_Dpn_(k=5)', 'knn_count_EdU_(k=5)',
       'knn_count_Lobe_(k=5)', 'knn_count_Scrib_(k=5)',
       'knn_mean_distance_Dpn_(k=5)', 'knn_mean_distance_EdU_(k=5)',
       'knn_mean_distance_Lobe_(k=5)', 'knn_mean_distance_Scrib_(k=5)',
       'knn_nearest_neighbor_distance_Dpn_(k=5)',
       'knn_nearest_neighbor_distance_EdU_(k=5)',
       'knn_nearest_neighbor_distance_Lobe_(k=5)',
       'knn_nearest_neighbor_distance_Scrib_(k=5)',
 

In [33]:
# reorder columns
cols = list(features.columns.values)
print (cols)
basics = ["label", "object_name", "channel","lineage Lobe"] 
positives = [c for c in cols if "+" in c] 
counts = [c for c in cols if "count" in c]
lineage = []#[c for c in cols if "lineage" in c]
rest = [c for c in cols if c not in (basics+positives+counts+lineage)]
reordered_cols = basics + counts + positives + lineage + rest
features = features[reordered_cols]
features[basics+positives].groupby(["lineage Lobe", "channel","EdU +"]).count()

#features[["channel", "object_name", "lineage Lobe"]].groupby(["lineage Lobe", "channel"]).count()
#features[(features["channel"]=="Dpn") & (~features["count EdU"].isna())]["count EdU"]

['bbox-start-z', 'bbox-start-y', 'bbox-start-x', 'bbox-end-z', 'bbox-end-y', 'bbox-end-x', 'centroid-z', 'centroid-y', 'centroid-x', 'label', 'aspect_ratio-yz', 'aspect_ratio-xz', 'aspect_ratio-xy', 'aspect_ratio', 'cross_sectional_area-yz', 'cross_sectional_area-xz', 'cross_sectional_area-xy', 'volume', 'channel', 'stack_id', 'slice_id', 'object_name', 'count Dpn', 'count EdU', 'count Lobe', 'count Scrib', 'lineage Brain', 'lineage Dpn', 'lineage EdU', 'lineage Lobe', 'lineage Scrib', 'knn_count_Dpn_(k=5)', 'knn_count_EdU_(k=5)', 'knn_count_Lobe_(k=5)', 'knn_count_Scrib_(k=5)', 'knn_mean_distance_Dpn_(k=5)', 'knn_mean_distance_EdU_(k=5)', 'knn_mean_distance_Lobe_(k=5)', 'knn_mean_distance_Scrib_(k=5)', 'knn_nearest_neighbor_distance_Dpn_(k=5)', 'knn_nearest_neighbor_distance_EdU_(k=5)', 'knn_nearest_neighbor_distance_Lobe_(k=5)', 'knn_nearest_neighbor_distance_Scrib_(k=5)', 'knn_nearest_neighbor_id_Dpn_(k=5)', 'knn_nearest_neighbor_id_EdU_(k=5)', 'knn_nearest_neighbor_id_Lobe_(k=5)', 

label  object_name  Dpn +  Lobe +  Scrib +
lineage Lobe channel EdU +                                            
1.0          Dpn     False     65           65     65      65       65
                     True      28           28     28      28       28
             EdU     False    173          173    173     173      173
             Scrib   False    139          139    139     139      139
                     True      12           12     12      12       12
2.0          Dpn     False     69           69     69      69       69
                     True      23           23     23      23       23
             EdU     False    187          187    187     187      187
             Scrib   False    126          126    126     126      126
                     True      12           12     12      12       12

# Hierarchical label decomposition

# View in Napari

In [24]:
import napari
viewer = napari.Viewer()

In [27]:
scale = metadata["physical_pixel_sizes"]
channel_colors = ("green", "blue", "red")
nimg = img#np.expand_dims(img, axis=0)

# add labels, colormap={1: 'yellow', 2: 'cyan'},
for meta, l in zip(combined_metadata,combined_labels):#, combined_measurements):
    ch = "".join(meta.get("channel_names"))
    ch_features = features.loc[features["channel"]==ch]
    new_m = ch_features.copy().reset_index() #m.copy().reset_index()
    background = pd.DataFrame({c: [0] if c=="label" else [np.nan] for c in new_m.columns.to_list()})
    new_m = pd.concat([background, new_m], ignore_index=True)
    viewer.add_labels(l, name=f"{ch}-Labels", features=new_m, scale=scale, blending="translucent_no_depth", opacity=0.3, rendering="translucent")

# add cell labels for each channel
ch_images, ch_metadata = unstack_image(img, metadata=metadata, axis="C", strict=False)
for name, c_img, color in zip(metadata["channel_names"], ch_images, channel_colors):
    viewer.add_image(c_img, name=f"{name}", scale=scale, colormap=color, blending="additive", visible=False)
    


2026-07-06 16:00:03,040 - INFO - Unstacked image data along C axis with index 0. Data shapes: [(93, 512, 512), (93, 512, 512), (93, 512, 512)]


In [28]:
for ch in np.unique(features["channel"]):
    ch_features = features.loc[features["channel"]==ch]["label"]
    ch_viewer = np.unique(viewer.layers[f"{ch}-Labels"].data)
    print (f"{ch}, Features: {len(np.unique(ch_features))}, {len(ch_features)}, Viewer: {len(ch_viewer)}, {np.max(ch_viewer)+1}")
    

Brain, Features: 1, 1, Viewer: 2, 2
Dpn, Features: 185, 185, Viewer: 205, 205
EdU, Features: 360, 360, Viewer: 660, 660
Lobe, Features: 2, 2, Viewer: 3, 3
Scrib, Features: 289, 289, Viewer: 458, 458


# Visualize graph

In [29]:
def remove_edu(node):
    print (type(node))
    return True #dag.nodes[node].channel!="EdU"

In [30]:
import networkx as nx
dag = measurements["containment_graph"]

non_edu_nodes = [node for node, attr in dag.nodes(data=True) if attr.get("channel") not in ["EdU", "Scrib"]]


ndag = dag.subgraph(non_edu_nodes)

In [31]:
from vistiq.graph import nodes_to_layer, edges_to_layer

points = nodes_to_layer(
    ndag,
    "hierarchy nodes",
    node_face_color="EdU +",
    node_face_colormap=["magenta", "green", "red", "blue", "yellow"],
    #node_border_width=3,
    node_border_color="white",
    node_symbol="lineage Lobe",
    node_opacity=0.9,
    node_size=12,
    node_properties=features,
    editable=False,
)
vectors = edges_to_layer(
    ndag,
    "hierarchy edges",
    edge_color="lineage Lobe",
    edge_colormap="magma",
    edge_thickness=1.5,
    edge_opacity=0.35,
    editable=False,
)
viewer.add_layer(points)
viewer.add_layer(vectors)

<Vectors layer 'hierarchy edges' at 0x450fd2d50>